# Práctica 2 · Memoria y cuantización

**¿Cabe este modelo en tu GPU con el contexto que quieres usar?**

Vas a predecir y medir qué cambia al ejecutar **Qwen2.5-3B-Instruct** en **F16** y en **Q4_0** (pesos a 4 bits por bloques). En Colab, elige **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4**. Ejecuta las celdas en orden y haz una pausa en cada pregunta antes de ver la medición.

## 1. ¿Cuánta VRAM tienes disponible?

Mira el total y la memoria libre. **¿Cuánta VRAM podrías usar ahora mismo?** La sesión puede partir con memoria ya ocupada.

In [ ]:
import csv
import math
import os
import re
import shutil
import subprocess as sp
import tempfile
import time
from pathlib import Path

import matplotlib.pyplot as plt

if not shutil.which("nvidia-smi"):
    raise RuntimeError("Activa una GPU T4 en Colab y vuelve a ejecutar la celda.")

def leer_gpu():
    salida = sp.check_output(
        ["nvidia-smi", "-i", "0", "--query-gpu=name,memory.total,memory.used,memory.free",
         "--format=csv,noheader,nounits"],
        text=True, timeout=10,
    )
    nombre, total, usada, libre = next(csv.reader(salida.splitlines(), skipinitialspace=True))
    return {"nombre": nombre, "total": int(total), "usada": int(usada), "libre": int(libre)}

def dibujar_vram(filas, total_mib):
    fig, ax = plt.subplots(figsize=(10, max(2, 0.85 * len(filas) + 1.2)))
    for posicion, (nombre, segmentos) in enumerate(filas):
        inicio = 0
        for etiqueta, mib, color in segmentos:
            ancho = mib / 1024
            ax.barh(posicion, ancho, left=inicio, color=color, height=0.48,
                    label=etiqueta if posicion == 0 else "_nolegend_")
            if ancho >= 0.85:
                tinta = "#ffffff" if color in ("#28486b", "#126860") else "#142638"
                ax.text(inicio + ancho / 2, posicion, f"{ancho:.2f}",
                        va="center", ha="center", color=tinta, fontsize=11)
            inicio += ancho
    ax.set_yticks(range(len(filas)), [nombre for nombre, _ in filas])
    ax.invert_yaxis()
    ax.set_xlim(0, max(total_mib / 1024, ax.get_xlim()[1]))
    ax.set_xlabel(f"GiB · capacidad reportada: {total_mib / 1024:.2f} GiB")
    ax.axvline(total_mib / 1024, color="#142638", linestyle=":", linewidth=1)
    ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1), ncols=3, frameon=False)
    plt.show()
    plt.close(fig)

gpu_inicial = leer_gpu()
print(f'{gpu_inicial["nombre"]} · {gpu_inicial["libre"] / 1024:.2f} GiB libres')
dibujar_vram([("Antes de cargar", [
    ("Ocupada", gpu_inicial["usada"], "#28486b"),
    ("Libre", gpu_inicial["libre"], "#d9e6ee"),
])], gpu_inicial["total"])

## 2. Predice el tamaño de los pesos

El modelo tiene **3.090 millones de parámetros**. Para estimar, multiplica los parámetros por los bytes de cada peso y divide por **2³⁰** para pasar a GiB.

| Formato | Pista |
|---|---:|
| F16 | 2 bytes por peso |
| 4 bits ideales | 0,5 bytes por peso |

**¿Cuánto ocuparía cada uno? ¿Qué margen quedaría en tu GPU?** Escribe tus dos estimaciones y dibújalas antes de descargar el modelo. Usa punto para los decimales.

In [ ]:
def dibujar_estimaciones(f16_gib, q4_gib):
    filas = []
    for nombre, gib in [("F16", f16_gib), ("4 bits", q4_gib)]:
        if not math.isfinite(gib) or gib < 0:
            raise ValueError("Introduce una estimación en GiB mayor o igual a cero.")
        mib = gib * 1024
        margen = gpu_inicial["libre"] - mib
        print(f"{nombre}: {abs(margen) / 1024:.2f} GiB "
              f"{'libres en la estimación' if margen >= 0 else 'por encima de lo disponible'}")
        filas.append((nombre, [
            ("Ocupación inicial", gpu_inicial["usada"], "#28486b"),
            ("Pesos estimados", mib, "#126860"),
            ("Margen estimado", max(0, margen), "#d9e6ee"),
        ]))
    dibujar_vram(filas, gpu_inicial["total"])

estimacion_f16 = float(input("Tu estimación F16 (GiB): "))
estimacion_q4 = float(input("Tu estimación a 4 bits (GiB): "))
dibujar_estimaciones(estimacion_f16, estimacion_q4)

## 3. Carga y mide

Descarga los dos GGUF del mismo modelo: **F16 (6,18 GB)** y **Q4_0 (1,83 GB)**, ya cuantizado. La primera descarga puede tardar varios minutos. Se usan los binarios CUDA de llama.cpp **b11695**, la versión de la prueba en T4.

In [ ]:
from huggingface_hub import hf_hub_download

version = "b11695"
paquete = f"llama-{version}-bin-ubuntu-cuda-12.8-x64.tar.gz"
base_url = f"https://github.com/ggml-org/llama.cpp/releases/download/{version}"
destino = Path(f"/content/llama-{version}-cuda")
destino.mkdir(exist_ok=True)

for archivo in (paquete, f"cudart-{paquete}"):
    ruta = destino / archivo
    if not ruta.exists():
        temporal = ruta.with_suffix(".part")
        print(f"Descargando {archivo}…", flush=True)
        sp.run(["curl", "-fLsS", "--retry", "2", "-o", str(temporal),
                f"{base_url}/{archivo}"], check=True)
        temporal.replace(ruta)
    sp.run(["tar", "-xzf", str(ruta), "-C", str(destino)], check=True)

bin_dir = next(destino.rglob("llama-cli")).parent
entorno = os.environ.copy()
bibliotecas = sorted({str(p.parent) for p in destino.rglob("*.so*")})
if entorno.get("LD_LIBRARY_PATH"):
    bibliotecas.append(entorno["LD_LIBRARY_PATH"])
entorno["LD_LIBRARY_PATH"] = ":".join(bibliotecas)

dispositivos = sp.check_output(
    [str(bin_dir / "llama-cli"), "--list-devices"],
    env=entorno, stderr=sp.STDOUT, text=True, timeout=30,
)
print(dispositivos)
if not re.search(r"CUDA0:\s+", dispositivos):
    raise RuntimeError("llama.cpp no ha detectado CUDA. Revisa la GPU asignada por Colab.")

archivos = {
    "F16": "Qwen2.5-3B-Instruct-f16.gguf",
    "Q4_0": "Qwen2.5-3B-Instruct-Q4_0.gguf",
}
modelos = {}
for nombre, archivo in archivos.items():
    print(f"Descargando {nombre}…", flush=True)
    modelos[nombre] = Path(hf_hub_download(
        repo_id="bartowski/Qwen2.5-3B-Instruct-GGUF",
        filename=archivo,
        revision="f302c64a2269a69fb27b2f9473b362f5bb8e78d8",
    ))
for nombre, archivo in modelos.items():
    print(f"{nombre}: {archivo.stat().st_size / 2**30:.2f} GiB en disco")

Ahora mide **una variante cada vez**. La gráfica muestra la ocupación global de la GPU en el **pico observado por muestreo** durante una respuesta corta. El incremento incluye carga, caché y buffers. Las respuestas se guardan para el cierre.

**¿Cuánto se parece ese incremento a tu estimación de los pesos?**

In [ ]:
PROMPTS = {
    "Extracción JSON": (
        'Devuelve únicamente un objeto JSON con las claves "ciudad", "dia" y "asistentes". '
        'El día debe ser una cadena y el número de asistentes un entero.\n'
        'Texto: El 14 de mayo se celebró en Zaragoza un taller al que asistieron 27 personas.'
    ),
    "Resumen con dos datos": (
        "Resume en una sola frase el aviso. Conserva la nueva hora y la nueva sala.\n"
        "Aviso: El taller estaba previsto a las 10:30 en la sala B. "
        "Por mantenimiento, se traslada a la sala C y empieza a las 11:15."
    ),
}

def ejecutar_chat(modelo, prompt, medir=False):
    comando = [str(bin_dir / "llama-cli"), "-m", str(modelo), "-ngl", "999",
               "-c", "512", "-n", "96", "-ctk", "f16", "-ctv", "f16",
               "--temp", "0", "--single-turn", "-p", prompt]
    muestras = []
    with tempfile.TemporaryFile(mode="w+b") as salida:
        with sp.Popen(comando, stdin=sp.DEVNULL, stdout=salida, stderr=sp.STDOUT,
                      env=entorno) as proceso:
            try:
                limite = time.monotonic() + 240
                while medir and proceso.poll() is None:
                    if time.monotonic() >= limite:
                        raise TimeoutError("llama-cli no terminó en cuatro minutos.")
                    muestra = leer_gpu()
                    muestras.append(muestra)
                    time.sleep(0.2)
                proceso.wait(timeout=max(1, limite - time.monotonic()))
            finally:
                if proceso.poll() is None:
                    proceso.kill()
                    proceso.wait()
        salida.seek(0)
        texto = salida.read().decode("utf-8", errors="replace")
    if proceso.returncode:
        raise RuntimeError(texto[-4000:])
    _, marca, respuesta = texto.partition(prompt)
    if not marca:
        raise RuntimeError(f"No se encontró el turno en la salida:\n{texto[-2000:]}")
    respuesta = re.split(r"\n\s*\[\s*Prompt:|\n\s*Exiting\.\.\.", respuesta, maxsplit=1)[0]
    return respuesta.strip(), max(muestras, key=lambda dato: dato["usada"]) if muestras else None

cargas = {}
respuestas = {}
for nombre, modelo in modelos.items():
    print(f"Probando {nombre}…", flush=True)
    antes = leer_gpu()
    respuestas[nombre] = {}
    for indice, (tarea, prompt) in enumerate(PROMPTS.items()):
        respuesta, muestra = ejecutar_chat(modelo, prompt, medir=(indice == 0))
        respuestas[nombre][tarea] = respuesta
        if indice == 0:
            if muestra is None:
                raise RuntimeError("No se obtuvo ninguna muestra de memoria durante el turno.")
            cargas[nombre] = (antes, muestra)

filas = []
for nombre, (antes, pico) in cargas.items():
    base = min(antes["usada"], pico["usada"])
    filas.append((nombre, [
        ("Ocupación previa", base, "#28486b"),
        ("Incremento observado", pico["usada"] - base, "#126860"),
        ("Libre en esa muestra", pico["libre"], "#d9e6ee"),
    ]))
    print(f'{nombre}: {pico["libre"] / 1024:.2f} GiB libres en el pico observado')
dibujar_vram(filas, gpu_inicial["total"])

## 4. ¿Qué cambia con un prompt más largo?

Vamos a procesar entradas sintéticas de **256** y **4.096 tokens**, con un contexto ajustado a cada longitud. En ambas variantes, **K y V se guardan en FP16**. llama.cpp reserva la caché al configurar el contexto.

**Si la entrada es 16 veces más larga, ¿qué esperas que pase con el buffer del modelo, la KV cache y el buffer de cómputo? ¿Y si solo cambias los pesos de F16 a Q4_0?**

In [ ]:
LONGITUDES = (256, 4096)
NUEVOS_TOKENS = 32

def medir_variante(modelo):
    resultado = sp.run(
        [str(bin_dir / "llama-bench"), "-m", str(modelo), "-ngl", "999",
         "-ctk", "f16", "-ctv", "f16", "-p", ",".join(map(str, LONGITUDES)),
         "-n", str(NUEVOS_TOKENS), "-r", "3", "-v"],
        env=entorno, capture_output=True, text=True, timeout=300,
    )
    if resultado.returncode:
        print(resultado.stderr[-4000:])
        resultado.check_returncode()

    registro = resultado.stderr + "\n" + resultado.stdout
    capas = re.search(r"offloaded (\d+)/(\d+) layers to GPU", registro)
    pesos = re.search(r"CUDA0 model buffer size\s*=\s*([\d.]+) MiB", registro)
    reservas = re.findall(
        r"CUDA0 KV buffer size\s*=\s*([\d.]+) MiB"
        r".*?CUDA0 compute buffer size\s*=\s*([\d.]+) MiB",
        registro, flags=re.S,
    )
    velocidades = {}
    for linea in resultado.stdout.splitlines():
        if not linea.startswith("|"):
            continue
        columnas = [valor.strip() for valor in linea.strip("|").split("|")]
        if len(columnas) == 7 and re.fullmatch(r"(pp|tg)\d+", columnas[5]):
            velocidades[columnas[5]] = float(columnas[6].split()[0])
    pruebas = [f"pp{n}" for n in LONGITUDES] + [f"tg{NUEVOS_TOKENS}"]
    if (not capas or capas[1] != capas[2] or not pesos
            or len(reservas) != len(pruebas) or not all(p in velocidades for p in pruebas)):
        print(resultado.stdout)
        raise RuntimeError("No se pudo verificar la carga CUDA o leer las reservas del benchmark.")
    return {
        "pesos": float(pesos[1]),
        "capas": f"{capas[1]}/{capas[2]}",
        "contextos": dict(zip(LONGITUDES, [tuple(map(float, fila)) for fila in reservas])),
        "velocidades": velocidades,
    }

mediciones = {}
for nombre, modelo in modelos.items():
    print(f"Midiendo {nombre} en CUDA…", flush=True)
    mediciones[nombre] = medir_variante(modelo)
    dato = mediciones[nombre]
    print(f'{nombre} · {dato["capas"]} capas en GPU · '
          f'{dato["pesos"] / 1024:.2f} GiB en el buffer del modelo')

ahorro = 1 - mediciones["Q4_0"]["pesos"] / mediciones["F16"]["pesos"]
print(f"Reducción del buffer del modelo en GPU: {ahorro:.0%}")
print("Cálculo ideal de los pesos: 5,76 GiB en F16 y 1,44 GiB a 4 bits.")

Q4_0 usa escalas por bloque y el GGUF puede conservar algunos tensores con otra precisión. Compara la estimación ideal, el tamaño en disco y el buffer del modelo en CUDA. Cada cifra responde a una pregunta distinta.

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(10, 3.5), sharex=True, sharey=True)
for ax, (nombre, dato) in zip(ejes, mediciones.items()):
    kv, computo = zip(*(dato["contextos"][n] for n in LONGITUDES))
    etiquetas = [f"{n:,}".replace(",", ".") for n in LONGITUDES]
    barras_kv = ax.barh(etiquetas, kv, color="#126860", label="KV cache")
    barras_computo = ax.barh(etiquetas, computo, left=kv, color="#d9e6ee", label="Cómputo")
    ax.bar_label(barras_kv, labels=[f"{v:g}" if v >= 30 else "" for v in kv], padding=2,
                 label_type="center", color="#ffffff", fontsize=9)
    for barra, valor in zip(barras_kv, kv):
        if valor < 30:
            ax.text(valor + 4, barra.get_y() + barra.get_height() / 2, f"{valor:g}",
                    va="center", color="#142638", fontsize=9)
    ax.bar_label(barras_computo, labels=[f"{v:g}" for v in computo], padding=4)
    ax.set_title(nombre)
    ax.set_xlabel("Reserva en GPU (MiB)")
ejes[0].invert_yaxis()
ejes[0].set_ylabel("Tokens de contexto")
ejes[1].legend(frameon=False)
ejes[0].set_xlim(0, max(sum(par) for d in mediciones.values()
                        for par in d["contextos"].values()) * 1.2)
fig.tight_layout()
plt.show()
plt.close(fig)

## 5. ¿Sale gratis cuantizar?

**¿Esperas la misma calidad? ¿Cuál crees que procesará antes la entrada? ¿Cuál generará antes la salida?**

- **Extracción JSON.** Comprueba ciudad, fecha y número de asistentes. ¿Puedes parsear la respuesta completa como JSON?
- **Resumen.** Comprueba que conserva **11:15** y **sala C** en una frase.

In [ ]:
import html
import json

from IPython.display import HTML, display

for tarea in PROMPTS:
    bloques = []
    for nombre in modelos:
        respuesta = respuestas[nombre][tarea]
        bloques.append(f"<div style='flex:1;min-width:240px'><strong>{nombre}</strong>"
                       f"<pre style='white-space:pre-wrap'>{html.escape(respuesta)}</pre></div>")
    display(HTML(f"<h4>{html.escape(tarea)}</h4>"
                 f"<div style='display:flex;flex-wrap:wrap;gap:1rem'>{''.join(bloques)}</div>"))

esperado = {"ciudad": "Zaragoza", "dia": "14 de mayo", "asistentes": 27}
for nombre in modelos:
    try:
        objeto = json.loads(respuestas[nombre]["Extracción JSON"])
        veredicto = ("JSON válido y datos exactos" if objeto == esperado
                     else "JSON válido con diferencias respecto a los datos esperados")
    except json.JSONDecodeError:
        veredicto = "la respuesta completa no es JSON válido"
    print(f"{nombre}: {veredicto}")

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(10, 3.4))
for ax, prueba, titulo in zip(
    ejes, (f"pp{LONGITUDES[-1]}", f"tg{NUEVOS_TOKENS}"),
    ("Prefill · 4.096 tokens de entrada", "Decode · 32 tokens de salida"),
):
    valores = [d["velocidades"][prueba] for d in mediciones.values()]
    barras = ax.barh(list(mediciones), valores, color=["#28486b", "#126860"])
    ax.bar_label(barras, labels=[f"{v:,.1f}".replace(",", " ").replace(".", ",")
                                 for v in valores], padding=5)
    ax.invert_yaxis()
    ax.set_xlim(0, max(valores) * 1.24)
    ax.set_xlabel("Tokens/s · media de 3 repeticiones")
    ax.set_title(titulo)
fig.tight_layout()
plt.show()
plt.close(fig)

**¿Qué versión usarías para tu tarea con la VRAM, las respuestas y los tiempos que acabas de observar?**

- En **prefill**, el motor procesa los tokens de entrada en paralelo y reutiliza los pesos. En **decode**, produce un token por paso y vuelve a leer gran parte de los pesos. Pesos más compactos pueden acelerar esa fase cuando el ancho de banda domina.
- Las barras vienen de dos pruebas separadas de llama-bench. La de decode empieza con un contexto corto. Las respuestas de arriba permiten comparar estas tareas concretas.
- La precisión de la KV cache también se puede cambiar por separado. Reducirla ahorra memoria para contextos largos y puede afectar a la calidad y al tiempo.

<small>Fuentes · <a href="https://huggingface.co/Qwen/Qwen2.5-3B-Instruct">Modelo</a> · <a href="https://huggingface.co/bartowski/Qwen2.5-3B-Instruct-GGUF">GGUF</a> · <a href="https://github.com/ggml-org/llama.cpp/releases/tag/b11695">Binarios</a> · <a href="https://github.com/ggml-org/llama.cpp/blob/master/tools/llama-bench/README.md">llama-bench</a> · <a href="https://developer.nvidia.com/blog/mastering-llm-techniques-inference-optimization/">Prefill y decode</a></small>